<a href="https://colab.research.google.com/github/aymensrihi/deep-learning-projects/blob/main/untestedpowerfull.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
Swin-SPSD with Learnable Stage Attention (LSA-SPSD) - CORRECTED VERSION
✅ FIX 1: Proper KL divergence direction (teacher || student)
✅ FIX 2: Probability-level mixing (not logit-level)
✅ FIX 3: Entropy regularization to prevent premature collapse
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import einsum
from torchvision import transforms
import numpy as np
import random
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from einops import rearrange, repeat
from tqdm import tqdm
import time
from collections import deque

# ============================================================================
# SWIN TRANSFORMER COMPONENTS (unchanged)
# ============================================================================

class CyclicShift(nn.Module):
    def __init__(self, displacement):
        super().__init__()
        self.displacement = displacement

    def forward(self, x):
        return torch.roll(x, shifts=(self.displacement, self.displacement), dims=(1, 2))


class Residual(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(x, **kwargs) + x


class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(self.norm(x), **kwargs)


class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, dim),
        )

    def forward(self, x):
        return self.net(x)


def create_mask(window_size, displacement, upper_lower, left_right):
    mask = torch.zeros(window_size ** 2, window_size ** 2)

    if upper_lower:
        mask[-displacement * window_size:, :-displacement * window_size] = float('-inf')
        mask[:-displacement * window_size, -displacement * window_size:] = float('-inf')

    if left_right:
        mask = rearrange(mask, '(h1 w1) (h2 w2) -> h1 w1 h2 w2', h1=window_size, h2=window_size)
        mask[:, -displacement:, :, :-displacement] = float('-inf')
        mask[:, :-displacement, :, -displacement:] = float('-inf')
        mask = rearrange(mask, 'h1 w1 h2 w2 -> (h1 w1) (h2 w2)')

    return mask


def get_relative_distances(window_size):
    indices = torch.tensor(np.array([[x, y] for x in range(window_size) for y in range(window_size)]))
    distances = indices[None, :, :] - indices[:, None, :]
    return distances


class WindowAttention(nn.Module):
    def __init__(self, dim, heads, head_dim, shifted, window_size, relative_pos_embedding):
        super().__init__()
        inner_dim = head_dim * heads

        self.heads = heads
        self.scale = head_dim ** -0.5
        self.window_size = window_size
        self.relative_pos_embedding = relative_pos_embedding
        self.shifted = shifted

        if self.shifted:
            displacement = window_size // 2
            self.cyclic_shift = CyclicShift(-displacement)
            self.cyclic_back_shift = CyclicShift(displacement)
            self.upper_lower_mask = nn.Parameter(create_mask(window_size=window_size, displacement=displacement,
                                                             upper_lower=True, left_right=False), requires_grad=False)
            self.left_right_mask = nn.Parameter(create_mask(window_size=window_size, displacement=displacement,
                                                            upper_lower=False, left_right=True), requires_grad=False)

        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias=False)

        if self.relative_pos_embedding:
            self.relative_indices = get_relative_distances(window_size) + window_size - 1
            self.pos_embedding = nn.Parameter(torch.randn(2 * window_size - 1, 2 * window_size - 1))
        else:
            self.pos_embedding = nn.Parameter(torch.randn(window_size ** 2, window_size ** 2))

        self.to_out = nn.Linear(inner_dim, dim)

    def forward(self, x):
        if self.shifted:
            x = self.cyclic_shift(x)

        b, n_h, n_w, _, h = *x.shape, self.heads

        qkv = self.to_qkv(x).chunk(3, dim=-1)
        nw_h = n_h // self.window_size
        nw_w = n_w // self.window_size

        q, k, v = map(
            lambda t: rearrange(t, 'b (nw_h w_h) (nw_w w_w) (h d) -> b h (nw_h nw_w) (w_h w_w) d',
                                h=h, w_h=self.window_size, w_w=self.window_size), qkv)

        dots = einsum('b h w i d, b h w j d -> b h w i j', q, k) * self.scale

        if self.relative_pos_embedding:
            dots += self.pos_embedding[self.relative_indices[:, :, 0], self.relative_indices[:, :, 1]]
        else:
            dots += self.pos_embedding

        if self.shifted:
            dots[:, :, -nw_w:] += self.upper_lower_mask
            dots[:, :, nw_w - 1::nw_w] += self.left_right_mask

        attn = dots.softmax(dim=-1)

        out = einsum('b h w i j, b h w j d -> b h w i d', attn, v)
        out = rearrange(out, 'b h (nw_h nw_w) (w_h w_w) d -> b (nw_h w_h) (nw_w w_w) (h d)',
                        h=h, w_h=self.window_size, w_w=self.window_size, nw_h=nw_h, nw_w=nw_w)
        out = self.to_out(out)

        if self.shifted:
            out = self.cyclic_back_shift(out)
        return out


class SwinBlock(nn.Module):
    def __init__(self, dim, heads, head_dim, mlp_dim, shifted, window_size, relative_pos_embedding):
        super().__init__()
        self.attention_block = Residual(PreNorm(dim, WindowAttention(dim=dim,
                                                                     heads=heads,
                                                                     head_dim=head_dim,
                                                                     shifted=shifted,
                                                                     window_size=window_size,
                                                                     relative_pos_embedding=relative_pos_embedding)))
        self.mlp_block = Residual(PreNorm(dim, FeedForward(dim=dim, hidden_dim=mlp_dim)))

    def forward(self, x):
        x = self.attention_block(x)
        x = self.mlp_block(x)
        return x


class PatchMerging(nn.Module):
    def __init__(self, in_channels, out_channels, downscaling_factor):
        super().__init__()
        self.downscaling_factor = downscaling_factor
        self.patch_merge = nn.Unfold(kernel_size=downscaling_factor, stride=downscaling_factor, padding=0)
        self.linear = nn.Linear(in_channels * downscaling_factor ** 2, out_channels)

    def forward(self, x):
        b, c, h, w = x.shape
        new_h, new_w = h // self.downscaling_factor, w // self.downscaling_factor
        x = self.patch_merge(x).view(b, -1, new_h, new_w).permute(0, 2, 3, 1)
        x = self.linear(x)
        return x


class StageModule(nn.Module):
    def __init__(self, in_channels, hidden_dimension, layers, downscaling_factor, num_heads, head_dim, window_size,
                 relative_pos_embedding):
        super().__init__()
        assert layers % 2 == 0, 'Stage layers need to be divisible by 2 for regular and shifted block.'

        self.patch_partition = PatchMerging(in_channels=in_channels, out_channels=hidden_dimension,
                                            downscaling_factor=downscaling_factor)

        self.layers = nn.ModuleList([])
        for _ in range(layers // 2):
            self.layers.append(nn.ModuleList([
                SwinBlock(dim=hidden_dimension, heads=num_heads, head_dim=head_dim, mlp_dim=hidden_dimension * 4,
                          shifted=False, window_size=window_size, relative_pos_embedding=relative_pos_embedding),
                SwinBlock(dim=hidden_dimension, heads=num_heads, head_dim=head_dim, mlp_dim=hidden_dimension * 4,
                          shifted=True, window_size=window_size, relative_pos_embedding=relative_pos_embedding),
            ]))

    def forward(self, x):
        x = self.patch_partition(x)
        for regular_block, shifted_block in self.layers:
            x = regular_block(x)
            x = shifted_block(x)
        return x.permute(0, 3, 1, 2)


class SwinTransformer_SPSD(nn.Module):
    def __init__(self, *, hidden_dim, layers, heads, channels=3, num_classes=1000, head_dim=32, window_size=7,
                 downscaling_factors=(4, 2, 2, 2), relative_pos_embedding=True):
        super().__init__()

        self.stage1 = StageModule(in_channels=channels, hidden_dimension=hidden_dim, layers=layers[0],
                                  downscaling_factor=downscaling_factors[0], num_heads=heads[0], head_dim=head_dim,
                                  window_size=window_size, relative_pos_embedding=relative_pos_embedding)
        self.stage2 = StageModule(in_channels=hidden_dim, hidden_dimension=hidden_dim * 2, layers=layers[1],
                                  downscaling_factor=downscaling_factors[1], num_heads=heads[1], head_dim=head_dim,
                                  window_size=window_size, relative_pos_embedding=relative_pos_embedding)
        self.stage3 = StageModule(in_channels=hidden_dim * 2, hidden_dimension=hidden_dim * 4, layers=layers[2],
                                  downscaling_factor=downscaling_factors[2], num_heads=heads[2], head_dim=head_dim,
                                  window_size=window_size, relative_pos_embedding=relative_pos_embedding)
        self.stage4 = StageModule(in_channels=hidden_dim * 4, hidden_dimension=hidden_dim * 8, layers=layers[3],
                                  downscaling_factor=downscaling_factors[3], num_heads=heads[3], head_dim=head_dim,
                                  window_size=window_size, relative_pos_embedding=relative_pos_embedding)

        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.AdaptiveAvgPool2d(1),
                nn.Flatten(),
                nn.LayerNorm(hidden_dim * (2 ** i)),
                nn.Linear(hidden_dim * (2 ** i), num_classes)
            )
            for i in range(4)
        ])

    def forward(self, img):
        stage_outputs = []
        x = self.stage1(img)
        stage_outputs.append(x)
        x = self.stage2(x)
        stage_outputs.append(x)
        x = self.stage3(x)
        stage_outputs.append(x)
        x = self.stage4(x)
        stage_outputs.append(x)

        predictions = [head(feat) for feat, head in zip(stage_outputs, self.heads)]
        return predictions


def swin_t_spsd(hidden_dim=96, layers=(2, 2, 6, 2), heads=(3, 6, 12, 24), **kwargs):
    return SwinTransformer_SPSD(hidden_dim=hidden_dim, layers=layers, heads=heads, **kwargs)


# ============================================================================
# DATASET CLASS
# ============================================================================

class DRDataset(Dataset):
    def __init__(self, root, domain_name, transform=None):
        self.root = os.path.join(root, domain_name)
        self.transform = transform
        self.classes = ['0', '1', '2', '3', '4']
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

        self.samples = []
        image_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.gif')

        for class_name in self.classes:
            class_dir = os.path.join(self.root, class_name)
            if not os.path.exists(class_dir):
                continue

            class_idx = self.class_to_idx[class_name]
            for img_name in os.listdir(class_dir):
                if img_name.lower().endswith(image_extensions):
                    self.samples.append((os.path.join(class_dir, img_name), class_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, target = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
            if self.transform:
                img = self.transform(img)
            return img, target
        except Exception as e:
            print(f"Error loading {path}: {e}")
            return self.__getitem__(random.randint(0, len(self) - 1))


# ============================================================================
# STAGE ATTENTION TRACKER
# ============================================================================

class StageAttentionTracker:
    """Tracks the evolution of learned stage attention weights during training."""
    def __init__(self):
        self.weight_history = []
        self.entropy_history = []

    def record_weights(self, weights):
        """Record normalized weights for this epoch"""
        weights_np = weights.detach().cpu().numpy()
        self.weight_history.append(weights_np.copy())

        # Calculate entropy: H = -sum(w * log(w))
        entropy = -np.sum(weights_np * np.log(weights_np + 1e-10))
        self.entropy_history.append(entropy)

    def get_statistics(self):
        """Get comprehensive statistics about learned attention"""
        if not self.weight_history:
            return {}

        current_weights = self.weight_history[-1]
        dominant_stage = np.argmax(current_weights)
        dominant_weight = current_weights[dominant_stage]

        max_entropy = np.log(4)
        current_entropy = self.entropy_history[-1]
        concentration = 1 - (current_entropy / max_entropy)

        return {
            'current_weights': current_weights,
            'dominant_stage': dominant_stage,
            'dominant_weight': dominant_weight,
            'entropy': current_entropy,
            'concentration': concentration,
            'weight_history': self.weight_history,
            'entropy_history': self.entropy_history
        }

    def print_evolution(self):
        """Print how weights evolved over training"""
        if not self.weight_history:
            return

        print("\n" + "="*80)
        print("Stage Attention Weight Evolution")
        print("="*80)
        print("  Epoch | Stage 1 | Stage 2 | Stage 3 | Stage 4 | Entropy | Focus")
        print("  " + "-"*78)

        for epoch, (weights, entropy) in enumerate(zip(self.weight_history, self.entropy_history), 1):
            dominant = np.argmax(weights)
            focus = f"S{dominant+1}"
            print(f"  {epoch:5d} | {weights[0]:6.3f}  | {weights[1]:6.3f}  | "
                  f"{weights[2]:6.3f}  | {weights[3]:6.3f}  | {entropy:6.3f}  | {focus}")

        print("="*80 + "\n")


# ============================================================================
# ✅ CORRECTED: LEARNABLE STAGE ATTENTION SPSD
# ============================================================================

class LearnableStageSPSD(nn.Module):
    """
    Learnable Stage Attention SPSD - CORRECTED VERSION

    Key Corrections:
    1. Proper KL direction: KL(teacher || student) where teacher = final stage
    2. Probability-level mixing (not logit-level)
    3. Entropy regularization to prevent premature collapse

    Mathematical formulation:
        p_i = softmax(z_i)  for each stage i
        p_adaptive = sum(w_i * p_i)  (probability mixture)
        L_RB = KL(p_final.detach() || p_adaptive)
        L_entropy = -sum(w_i * log(w_i))  (regularization)
        L_total = L_base + λ * L_RB + γ * L_entropy
    """
    def __init__(self, num_classes=5, hparams=None):
        super().__init__()
        if hparams is None:
            hparams = default_hparams()

        self.hparams = hparams
        self.lambda_ = hparams['RB_loss_weight']
        self.beta_T = hparams['alpha_T']
        self.n_steps = hparams['n_steps']
        self.step_count = 0
        self.n_classes = num_classes

        # ✅ Learnable stage attention weights (log-space)
        self.stage_weights = nn.Parameter(torch.zeros(4))

        # ✅ NEW: Entropy regularization scheduler
        self.gamma_start = hparams.get('entropy_weight_start', 0.1)
        self.gamma_end = hparams.get('entropy_weight_end', 0.0)

        self.attention_tracker = StageAttentionTracker()

        self.network = swin_t_spsd(
            hidden_dim=96,
            layers=(2, 2, 6, 2),
            heads=(3, 6, 12, 24),
            num_classes=num_classes,
            channels=3,
            window_size=7
        )

        self.optimizer = torch.optim.AdamW(
            list(self.network.parameters()) + [self.stage_weights],
            lr=hparams["lr"],
            weight_decay=hparams['weight_decay']
        )

    def get_normalized_weights(self):
        """Get current stage attention weights (normalized to sum to 1)"""
        return F.softmax(self.stage_weights, dim=0)

    def get_entropy_weight(self):
        """
        ✅ FIX 3: Annealed entropy regularization weight

        Starts high (encourage exploration) → Ends at 0 (allow specialization)
        γ_t = γ_start * (1 - t/T) + γ_end
        """
        progress = self.step_count / self.n_steps
        gamma_t = self.gamma_start * (1 - progress) + self.gamma_end
        return gamma_t

    def update(self, x, y):
        """✅ CORRECTED: Training step with proper KL, probability mixing, and entropy regularization"""
        # Progressive soft pseudo-labeling weight
        beta_t = self.beta_T * ((self.step_count + 1) / self.n_steps)
        beta_t = max(0.0, min(beta_t, self.beta_T))
        self.step_count += 1

        # Forward pass through all stages
        outputs = self.network(x)  # List of logits [z1, z2, z3, z4]
        z_final = outputs[-1]  # Final stage (teacher)

        # ✅ Compute normalized attention weights
        weights = self.get_normalized_weights()

        # ✅ FIX 2: PROBABILITY-LEVEL MIXING (not logit-level)
        # Convert all logits to probabilities first
        probs = [F.softmax(z_i, dim=1) for z_i in outputs]

        # Weighted mixture of probabilities
        p_adaptive = sum(w * probs[i] for i, w in enumerate(weights))
        p_final = probs[-1]  # Final stage probability

        # One-hot encoding for labels
        y_one_hot = torch.zeros(y.size(0), self.n_classes, device=x.device)
        y_one_hot.scatter_(1, y.unsqueeze(1), 1)

        # Soft pseudo-labels (only for distillation target, not for mixing)
        soft_p_final = beta_t * p_final + (1 - beta_t) * y_one_hot

        # ✅ FIX 1: CORRECT KL DIRECTION
        # KL(teacher || student) where:
        # - teacher = final stage (detached)
        # - student = adaptive mixture (gradients flow here)
        # This distills knowledge FROM final stage TO the mixture
        base_loss = F.cross_entropy(z_final, y)

        rb_loss = F.kl_div(
            torch.log(p_adaptive + 1e-10),  # Student (log-probabilities)
            soft_p_final.detach(),           # Teacher (detached)
            reduction='batchmean'
        )

        # ✅ FIX 3: ENTROPY REGULARIZATION
        # H = -sum(w_i * log(w_i))
        # Prevents premature collapse to single stage
        entropy = -torch.sum(weights * torch.log(weights + 1e-10))
        gamma_t = self.get_entropy_weight()
        entropy_loss = -gamma_t * entropy  # Negative because we want to maximize entropy early

        # Total loss
        loss = base_loss + self.lambda_ * rb_loss + entropy_loss

        # Optimization
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        return {
            'loss': loss.item(),
            'base_loss': base_loss.item(),
            'rb_loss': rb_loss.item(),
            'entropy_loss': entropy_loss.item(),
            'entropy': entropy.item(),
            'gamma_t': gamma_t,
            'beta_t': beta_t,
            'stage_weights': weights.detach().cpu().numpy(),
            'dominant_stage': weights.argmax().item()
        }

    def commit_epoch_stats(self):
        """Record stage weights at end of epoch"""
        weights = self.get_normalized_weights()
        self.attention_tracker.record_weights(weights)
        return self.attention_tracker.get_statistics()

    def get_attention_stats(self):
        """Get comprehensive attention statistics"""
        return self.attention_tracker.get_statistics()

    def predict(self, x):
        outputs = self.network(x)
        return outputs[-1]


# ============================================================================
# TRAINING UTILITIES
# ============================================================================

def default_hparams():
    return {
        'data_augmentation': True,
        'RB_loss_weight': 0.7,
        'alpha_T': 0.8,
        'n_steps': None,
        'lr': 5e-5,
        'weight_decay': 0.05,
        'batch_size': 32,
        # ✅ NEW: Entropy regularization parameters
        'entropy_weight_start': 0.1,  # Start with high entropy (exploration)
        'entropy_weight_end': 0.0      # End with zero (allow specialization)
    }


def get_transforms(augment=True):
    transform_list = [transforms.Resize((224, 224))]
    if augment:
        transform_list.extend([
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.4, contrast=0.4)
        ])
    transform_list.extend([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return transforms.Compose(transform_list)


@torch.no_grad()
def evaluate(model, data_loader, device):
    model.eval()
    correct = 0
    total = 0

    for x, y in data_loader:
        x, y = x.to(device), y.to(device)
        predictions = model.predict(x)
        _, predicted = torch.max(predictions, 1)
        correct += (predicted == y).sum().item()
        total += y.size(0)

    return correct / total if total > 0 else 0.0


def format_time(seconds):
    """Format seconds into readable time string"""
    if seconds < 60:
        return f"{seconds:.0f}s"
    elif seconds < 3600:
        mins = seconds / 60
        return f"{mins:.1f}m"
    else:
        hours = seconds / 3600
        return f"{hours:.1f}h"


def train_multi_source_dg(data_root, test_domain, num_epochs=10, hparams=None):
    """Training with Learnable Stage Attention (CORRECTED VERSION)"""
    if hparams is None:
        hparams = default_hparams()

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}\n")

    all_domains = ['aptos', 'eyepacs', 'messidor', 'messidor_2']
    train_domains = [d for d in all_domains if d != test_domain]

    print(f"Training domains: {train_domains}")
    print(f"Test domain: {test_domain}\n")

    train_transform = get_transforms(augment=True)
    test_transform = get_transforms(augment=False)

    train_datasets = [DRDataset(data_root, d, train_transform) for d in train_domains]
    test_dataset = DRDataset(data_root, test_domain, test_transform)

    train_loaders = [
        DataLoader(ds, batch_size=hparams['batch_size'], shuffle=True, num_workers=2)
        for ds in train_datasets
    ]
    test_loader = DataLoader(test_dataset, batch_size=hparams['batch_size'],
                            shuffle=False, num_workers=2)

    print(f"Training samples: {sum(len(ds) for ds in train_datasets)}")
    print(f"Test samples: {len(test_dataset)}\n")

    steps_per_epoch = max(len(loader) for loader in train_loaders)
    total_steps = steps_per_epoch * num_epochs
    hparams['n_steps'] = total_steps

    print("=" * 80)
    print("✅ CORRECTED LSA-SPSD:")
    print(f"  Steps per epoch: {steps_per_epoch}")
    print(f"  Total training steps: {total_steps}")
    print(f"  β progression: 0.0 → {hparams['alpha_T']}")
    print(f"  γ progression: {hparams['entropy_weight_start']} → {hparams['entropy_weight_end']}")
    print("  ✓ FIX 1: Proper KL(teacher || student)")
    print("  ✓ FIX 2: Probability-level mixing")
    print("  ✓ FIX 3: Entropy regularization (annealed)")
    print("=" * 80)
    print()

    model = LearnableStageSPSD(num_classes=5, hparams=hparams).to(device)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {total_params:,}\n")

    best_acc = 0.0
    training_start_time = time.time()

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        epoch_base_loss = 0
        epoch_rb_loss = 0
        epoch_entropy_loss = 0
        n_batches = 0

        epoch_start_time = time.time()

        train_iters = [iter(loader) for loader in train_loaders]
        max_batches = max(len(loader) for loader in train_loaders)

        epochs_left = num_epochs - epoch - 1
        pbar = tqdm(
            range(max_batches),
            desc=f"Epoch {epoch+1}/{num_epochs} ({epochs_left} left)",
            bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]'
        )

        for batch_idx in pbar:
            all_x, all_y = [], []
            for train_iter, loader in zip(train_iters, train_loaders):
                try:
                    x, y = next(train_iter)
                except StopIteration:
                    train_iter = iter(loader)
                    x, y = next(train_iter)
                all_x.append(x)
                all_y.append(y)

            x = torch.cat(all_x).to(device)
            y = torch.cat(all_y).to(device)

            step_vals = model.update(x, y)

            epoch_loss += step_vals['loss']
            epoch_base_loss += step_vals['base_loss']
            epoch_rb_loss += step_vals['rb_loss']
            epoch_entropy_loss += step_vals['entropy_loss']
            n_batches += 1

            progress_pct = ((epoch * max_batches + batch_idx + 1) / (num_epochs * max_batches)) * 100
            weights = step_vals['stage_weights']
            dominant_idx = step_vals['dominant_stage']

            pbar.set_postfix({
                'loss': f"{step_vals['loss']:.4f}",
                'β': f"{step_vals['beta_t']:.3f}",
                'H': f"{step_vals['entropy']:.3f}",
                'focus': f"S{dominant_idx+1}"
            })

        attention_stats = model.commit_epoch_stats()
        epoch_time = time.time() - epoch_start_time
        test_acc = evaluate(model, test_loader, device)

        elapsed_total = time.time() - training_start_time
        avg_epoch_time = elapsed_total / (epoch + 1)
        eta = avg_epoch_time * epochs_left

        print(f"\n{'='*80}")
        print(f"EPOCH {epoch+1}/{num_epochs} COMPLETE | {epochs_left} epochs remaining")
        print(f"{'='*80}")
        print(f"  Time: {format_time(epoch_time)} | Avg: {format_time(avg_epoch_time)} | ETA: {format_time(eta)}")
        print(f"  Loss: {epoch_loss/n_batches:.4f}")
        print(f"  Base Loss: {epoch_base_loss/n_batches:.4f}")
        print(f"  RB Loss: {epoch_rb_loss/n_batches:.4f}")
        print(f"  Entropy Loss: {epoch_entropy_loss/n_batches:.4f}")
        print(f"  Test Accuracy ({test_domain}): {test_acc*100:.2f}%")

        if attention_stats:
            weights = attention_stats['current_weights']
            dominant = attention_stats['dominant_stage']
            concentration = attention_stats['concentration']
            entropy = attention_stats['entropy']

            print(f"\n  🧠 Learned Stage Attention:")
            print(f"     Weights: S1={weights[0]:.3f}, S2={weights[1]:.3f}, "
                  f"S3={weights[2]:.3f}, S4={weights[3]:.3f}")
            print(f"     Dominant: Stage {dominant+1} ({weights[dominant]*100:.1f}%)")
            print(f"     Entropy: {entropy:.3f} (concentration: {concentration*100:.1f}%)")

        if test_acc > best_acc:
            best_acc = test_acc
            print(f"\n  ✓ New best accuracy!")
        print(f"{'='*80}\n")

    total_training_time = time.time() - training_start_time
    model.attention_tracker.print_evolution()

    print("\n" + "=" * 80)
    print("✅ TRAINING COMPLETE:")
    print(f"  Total training time: {format_time(total_training_time)}")
    print(f"  Best test accuracy: {best_acc*100:.2f}%")

    final_stats = model.get_attention_stats()
    if final_stats:
        weights = final_stats['current_weights']
        print(f"\n  Final Stage Importance:")
        for i, w in enumerate(weights):
            bar = "█" * int(w * 50)
            print(f"    Stage {i+1}: {w*100:5.1f}% {bar}")

    print("=" * 80)

    return model, best_acc


# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    try:
        import einops
    except ImportError:
        print("Installing einops...")
        import subprocess
        subprocess.check_call(['pip', 'install', 'einops', '-q'])
        print("✓ einops installed\n")

    print("=" * 80)
    print("LSA-SPSD: Learnable Stage Attention (CORRECTED)")
    print("✅ FIX 1: Proper KL(teacher || student) direction")
    print("✅ FIX 2: Probability-level mixing (not logits)")
    print("✅ FIX 3: Entropy regularization to prevent collapse")
    print("=" * 80)

    data_path = ""

    if not os.path.exists(data_path):
        print(f"\nERROR: Dataset not found at {data_path}")
        print("Please update the data_path variable")
        exit(1)

    print(f"\n✓ Dataset found at: {data_path}")
    print(f"✓ Domains: {sorted(os.listdir(data_path))}\n")

    test_domain = 'messidor_2'
    num_epochs = 10

    hparams = default_hparams()

    print(f"Configuration:")
    print(f"  Backbone: Swin-Tiny")
    print(f"  Test domain: {test_domain}")
    print(f"  Epochs: {num_epochs}")
    print(f"  Batch size: {hparams['batch_size']}")
    print(f"  Learning rate: {hparams['lr']}")
    print(f"  λ (RB weight): {hparams['RB_loss_weight']}")
    print(f"  β_T (max PSPL): {hparams['alpha_T']}")
    print(f"  γ_start (entropy reg): {hparams['entropy_weight_start']}")
    print(f"  γ_end (entropy reg): {hparams['entropy_weight_end']}\n")

    model, best_acc = train_multi_source_dg(
        data_root=data_path,
        test_domain=test_domain,
        num_epochs=num_epochs,
        hparams=hparams
    )

    print("\n" + "=" * 80)
    print(f"✓ Training complete!")
    print(f"✓ Best accuracy on {test_domain}: {best_acc*100:.2f}%")
    print("=" * 80)

    print("\n" + "=" * 80)
    print("Testing on all domains:")
    print("=" * 80)

    all_domains = ['aptos', 'eyepacs', 'messidor', 'messidor_2']
    test_transform = get_transforms(augment=False)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    results = {}
    for domain in all_domains:
        test_dataset = DRDataset(data_path, domain, test_transform)
        test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
        acc = evaluate(model, test_loader, device)
        results[domain] = acc
        marker = "✓" if domain != test_domain else "→"
        print(f"  {marker} {domain:15s}: {acc*100:.2f}%")

    avg_acc = sum(results.values()) / len(results)
    print(f"\n  Average accuracy: {avg_acc*100:.2f}%")
    print("=" * 80)

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (ipython-input-3648965718.py, line 763)